# Deep Reconstruction of SIRS Model

Savorgnan Enrico \
University of Trieste, Italy \
A. Y. 2024-2025

*Thesis in Epidemiology*: \
"A new model in behavioral epidemiology: the SIRS model with social distancing"

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchdiffeq import odeint

import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from tqdm.notebook import tqdm
import numpy as np

%matplotlib inline

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## 1. Real Data

In [3]:
def sirs_deriv(t, y, beta=10, gamma=0.1, theta=0.01, mu=0.01):
    S, I, R = y
    dS = -beta * S * I + theta * R - mu * S
    dI = beta * S * I - gamma * I - mu * I
    dR = gamma * I - theta * R - mu * R
    return torch.tensor([dS, dI, dR], device=device)

In [69]:
# Generate synthetic data for SIRS model
t = torch.linspace(0., 500., 501, device=device)
t_valid = torch.linspace(0., 1000., 1001, device=device)
y0 = torch.tensor([0.99, 0.01, 0.0], device=device)

gamma = 1 / 7  # recovery rate (1 week)
mu = 1 / 80 / 365  # mortality rate (80 years)
theta = 1 / 365  # immunity loss rate (1 year)
beta = 10 * (mu + gamma)  # infection rate

with torch.no_grad():
    true_traj = odeint(lambda tt, yy: sirs_deriv(tt, yy, beta=beta, gamma=gamma, theta=theta, mu=mu), y0, t)
    true_traj_valid = odeint(lambda tt, yy: sirs_deriv(tt, yy, beta=beta, gamma=gamma, theta=theta, mu=mu), y0, t_valid)

In [70]:
# Visualize the data
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=true_traj[:, 0].cpu().numpy(),
    mode='lines',
    name='S (Susceptible)',
    line=dict(color='blue')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=true_traj[:, 1].cpu().numpy(),
    mode='lines',
    name='I (Infected)',
    line=dict(color='red')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=true_traj[:, 2].cpu().numpy(),
    mode='lines',
    name='R (Recovered)',
    line=dict(color='green')
))


fig.update_layout(
    title='SIRS Model Dynamics',
    xaxis_title='Time (days)',
    yaxis_title='Population Proportion',
    legend_title='Compartments',
    template='plotly_white',
    width=1200,
    height=600
)
fig.show()

## 2. Architectures

In [6]:
# 2.1 ResNet
class ResNetDynamics(nn.Module):

    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64), nn.ReLU(),
            nn.Linear(64, 64), nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, y):
        return y + self.net(y)


# 2.2 Neural ODE
class ODEFunc(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 64), nn.Tanh(),
            nn.Linear(64, 64), nn.Tanh(),
            nn.Linear(64, dim),
        )

    def forward(self, t, y):
        return self.net(y)


# 2.3 Universal Differential Equations (UDE)
class UDEFunc(nn.Module):
    def __init__(self, dim, gamma, mu):
        super().__init__()
        # γ and μ are fixed parameters
        self.gamma = gamma
        self.mu = mu

        # β and θ are learnable parameters
        self.beta = nn.Parameter(torch.tensor(0.5))
        self.theta = nn.Parameter(torch.tensor(0.01))

        # residual network
        self.nn = nn.Sequential(
            nn.Linear(dim, 64),
            nn.ReLU(),
            nn.Linear(64, dim)
        )

    def forward(self, t, y):
        S, I, R = torch.unbind(y, dim=-1)
        # analytic part of the SIRS model
        dS = -self.beta * S * I - self.mu * S + self.theta * R
        dI = self.beta * S * I - self.gamma * I - self.mu * I
        dR = self.gamma * I - self.mu * R - self.theta * R
        f_analytic = torch.stack([dS, dI, dR], dim=-1)
        # residual term
        return f_analytic + self.nn(y)


## 3. Training & Evaluation

In [71]:
def train_model(odefunc, name, lr=1e-3, epochs=2000):
    """
    Method for training Neural ODEs and Universal Differential Equations (UDEs).
    """

    optimizer = optim.Adam(odefunc.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    pbar = tqdm(range(epochs), desc=f"Training {name}")

    losses = []

    for epoch in pbar:
        optimizer.zero_grad()
        pred = odeint(odefunc, y0, t)
        loss = loss_fn(pred, true_traj)
        loss.backward()

        optimizer.step()
        if epoch % 5 == 0:
            if epoch % 100 == 0:
                with torch.no_grad():
                    pred_valid = odeint(odefunc, y0, t_valid)
                    loss_valid = loss_fn(pred_valid, true_traj_valid)
                tqdm.write(
                    f"Epoch {epoch:04d}/{epochs:04d} \t|\t Loss {loss.item():.6f} \t|\t Validation Loss {loss_valid.item():.6f}")
            else:
                tqdm.write(f"Epoch {epoch:04d}/{epochs:04d} \t|\t Loss {loss.item():.6f}")
            losses.append(loss.item())

    return odeint(odefunc, y0, t).detach(), losses

In [72]:
def train_resnet_func(resnet, y0, t, true_traj, name, lr=1e-3, epochs=2000):
    """
    Method for training a ResNet model for dynamical systems.
    """

    optimizer = optim.Adam(resnet.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    pbar = tqdm(range(epochs), desc=f"Training {name}")

    best_loss = float('inf')
    best_model = None

    losses = []

    dt = t[1] - t[0]  # Assuming uniform time steps

    for epoch in pbar:
        optimizer.zero_grad()

        pred_list = [y0]
        current_y = y0
        for i in range(len(t) - 1):
            # ResNet learns y(t+dt) = F(y(t))
            current_y = resnet(current_y)
            pred_list.append(current_y)

        pred_traj = torch.stack(pred_list)
        loss = loss_fn(pred_traj, true_traj)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            pbar.set_description(f'Loss: {loss.item():.4f}')
            losses.append(loss.item())
            if loss.item() < best_loss:
                best_loss = loss.item()
                best_model = resnet.state_dict()

    print(f'Training {name} completato. Loss finale: {loss.item():.6f}')

    return pred_traj.detach(), losses

In [100]:
resnet = ResNetDynamics(dim=3).to(device)
pure_ode = ODEFunc(dim=3).to(device)
ude = UDEFunc(dim=3, gamma=1 / 7, mu=1 / 80 / 365).to(device)

epochs = 1000
lr = 0.001

In [86]:
traj_resnet, loss_resnet = train_resnet_func(resnet, y0, t, true_traj, "ResNet", lr=lr*0.1, epochs=epochs)
torch.save(resnet.state_dict(),
           f'deep_models/resnet/ EP {epochs} LR {lr} LOSS {loss_resnet[-1]:.4f} γ {gamma:.4f} μ {mu:.4f} θ {theta:.4f} TIME 500.pth')

Training ResNet:   0%|          | 0/1000 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [89]:
traj_neural, loss_neural = train_model(pure_ode, "NeuralODE", lr=lr*0.1, epochs=epochs)
torch.save(pure_ode.state_dict(),
           f'deep_models/neuralode/ EP {epochs} LR {lr} LOSS {loss_neural[-1]:.4f} γ {gamma:.4f} μ {mu:.4f} θ {theta:.4f}  TIME 500 .pth')

Training NeuralODE:   0%|          | 0/1000 [00:00<?, ?it/s]

Epoch 0000/1000 	|	 Loss 0.994760 	|	 Validation Loss 0.984732
Epoch 0005/1000 	|	 Loss 0.994760
Epoch 0010/1000 	|	 Loss 0.994760
Epoch 0015/1000 	|	 Loss 0.994760
Epoch 0020/1000 	|	 Loss 0.994760
Epoch 0025/1000 	|	 Loss 0.994760
Epoch 0030/1000 	|	 Loss 0.994760
Epoch 0035/1000 	|	 Loss 0.994760
Epoch 0040/1000 	|	 Loss 0.994760
Epoch 0045/1000 	|	 Loss 0.994760
Epoch 0050/1000 	|	 Loss 0.994760
Epoch 0055/1000 	|	 Loss 0.994760
Epoch 0060/1000 	|	 Loss 0.994760
Epoch 0065/1000 	|	 Loss 0.994760
Epoch 0070/1000 	|	 Loss 0.994760
Epoch 0075/1000 	|	 Loss 0.994760
Epoch 0080/1000 	|	 Loss 0.994760
Epoch 0085/1000 	|	 Loss 0.994760
Epoch 0090/1000 	|	 Loss 0.994760
Epoch 0095/1000 	|	 Loss 0.994760
Epoch 0100/1000 	|	 Loss 0.994760 	|	 Validation Loss 0.984732
Epoch 0105/1000 	|	 Loss 0.994760
Epoch 0110/1000 	|	 Loss 0.994760
Epoch 0115/1000 	|	 Loss 0.994760
Epoch 0120/1000 	|	 Loss 0.994760
Epoch 0125/1000 	|	 Loss 0.994760
Epoch 0130/1000 	|	 Loss 0.994760
Epoch 0135/1000 	|	 Loss

In [ ]:
traj_ude, loss_ude = train_model(ude, "UDE", epochs=500, lr=0.001)
torch.save(ude.state_dict(),
           f'deep_models/ude/ EP {epochs} LR {lr} LOSS {loss_ude[-1]:.4f} γ {gamma:.4f} μ {mu:.4f} θ {theta:.4f} TIME 500 .pth')

Training UDE:   0%|          | 0/500 [00:00<?, ?it/s]

## 4. Visualization of Results

In [32]:
# Carico i modelli
pure_ode.load_state_dict(torch.load(f'deep_models/neuralode/EP 2000 LR 0.001 LOSS 0.0001 γ 0.1429 μ 0.0000 θ 0.0027 .pth'))
ude.load_state_dict(torch.load(f'deep_models/ude/EP 2000 LR 0.001 LOSS 0.0000 γ 0.1429 μ 0.0000 θ 0.0027 .pth'))
resnet.load_state_dict(
    torch.load(f'deep_models/resnet/ EP 2000 LR {lr} LOSS {loss_resnet[-1]:.4f} γ {gamma:.4f} μ {mu:.4f} θ {theta:.4f} .pth'))

t = torch.linspace(0., 50., 51, device=device)
# Calcolo le traiettorie
traj_neural = odeint(pure_ode, y0, t).detach()
traj_ude = odeint(ude, y0, t).detach()

In [36]:
labels = ['S', 'I', 'R']
colors = {
    'Ground Truth': 'black',
    'NeuralODE': 'blue',
    'UDE': 'green',
    'ResNet': 'red'
}
line_styles = {
    'Ground Truth': 'dash',
    'NeuralODE': 'solid',
    'UDE': 'solid',
    'ResNet': 'solid'
}

# Crea una figura con 3 righe di subplot e un asse x condiviso
fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[f'Traiettoria per {label}(t)' for label in labels]
)

# Itera attraverso ogni stato (S, I, R) per popolare i subplot
for i in range(3):
    # Aggiungi la traccia per il Ground Truth
    fig.add_trace(go.Scatter(
        x=t.cpu(),
        y=true_traj[:, i].cpu(),
        name='Ground Truth',
        mode='lines',
        line=dict(color=colors['Ground Truth'], dash=line_styles['Ground Truth']),
        legendgroup='Ground Truth',  # Raggruppa per mostrare una sola volta in legenda
        showlegend=(i == 0)  # Mostra in legenda solo per il primo subplot
    ), row=i + 1, col=1)

    # Aggiungi la traccia per la Neural ODE
    fig.add_trace(go.Scatter(
        x=t.cpu(),
        y=traj_neural[:, i].cpu(),
        name='NeuralODE',
        mode='lines',
        line=dict(color=colors['NeuralODE'], dash=line_styles['NeuralODE']),
        legendgroup='NeuralODE',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # Aggiungi la traccia per la UDE
    fig.add_trace(go.Scatter(
        x=t.cpu(),
        y=traj_ude[:, i].cpu(),
        name='UDE',
        mode='lines',
        line=dict(color=colors['UDE'], dash=line_styles['UDE']),
        legendgroup='UDE',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # Aggiungi la traccia per la ResNet
    fig.add_trace(go.Scatter(
        x=t.cpu(),
        y=traj_resnet[:, i].cpu(),
        name='ResNet',
        mode='lines',
        line=dict(color=colors['ResNet'], dash=line_styles['ResNet']),
        legendgroup='ResNet',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

# Aggiorna il layout della figura
fig.update_layout(
    title_text="Confronto delle Traiettorie dei Modelli",
    height=800,  # Altezza simile al figsize
    width=1200,  # Larghezza simile al figsize
    legend_title_text='Modelli',
    template='plotly_white',
)

# Aggiorna gli assi
fig.update_xaxes(title_text="Tempo", row=3, col=1)  # Titolo asse x solo per l'ultimo grafico
fig.update_yaxes(title_text="S", row=1, col=1)
fig.update_yaxes(title_text="I", row=2, col=1)
fig.update_yaxes(title_text="R", row=3, col=1)



# Mostra il grafico
fig.show()

In [37]:
# predizioni future

future_t = torch.linspace(50., 200., 1001, device=device)
total_t = torch.linspace(0., 500., 2001, device=device)

# valori reali
true_future_traj = odeint(lambda tt, yy: sirs_deriv(tt, yy), y0, total_t)

# predizioni con Neural ODE
future_traj_neural = odeint(pure_ode, traj_neural[-1], future_t).detach()
# predizioni con UDE
future_traj_ude = odeint(ude, traj_ude[-1], future_t).detach()
# predizioni con ResNet


# --- Extend time for future prediction ---
t_future_np = np.arange(0., 200., 1.0)
t_future_tensor = torch.tensor(t_future_np, dtype=torch.float32).to(device)

# --- Generate predictions for the future ---
with torch.no_grad():
    # True data for comparison
    true_future_traj = odeint(lambda t, y: sirs_deriv(t, y, beta=beta, gamma=gamma, theta=theta, mu=mu), y0,
                              t_future_tensor)

    # Predictions with NODE and UDE (using the correct initial condition y0)
    pred_future_node = odeint(pure_ode, y0, t_future_tensor)
    pred_future_ude = odeint(ude, y0, t_future_tensor)

    # **Correct prediction with ResNet**
    pred_future_resnet_list = [y0]
    current_y = y0
    for i in range(len(t_future_np) - 1):
        current_y = resnet(current_y)  # Apply the model iteratively
        pred_future_resnet_list.append(current_y)
    pred_future_resnet = torch.stack(pred_future_resnet_list)

In [38]:
labels = ['S(t)', 'I(t)', 'R(t)']
colors = {
    'Ground Truth': 'black',
    'ResNet': 'red',
    'Neural ODE': 'blue',
    'UDE': 'green',
}

fig = make_subplots(
    rows=3, cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    subplot_titles=[f'Previsione Futura per {label}' for label in labels]
)

for i in range(3):
    # Ground Truth trace
    fig.add_trace(go.Scatter(
        x=t_future_np,
        y=true_future_traj[:, i].cpu().numpy(),
        name=f'{labels[i]} Ground Truth',
        mode='lines',
        line=dict(color=colors['Ground Truth'], width=3, dash='solid'),
        legendgroup='Ground Truth',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # ResNet trace
    fig.add_trace(go.Scatter(
        x=t_future_np,
        y=pred_future_resnet[:, i].cpu().numpy(),
        name=f'{labels[i]} ResNet',
        mode='lines',
        line=dict(color=colors['ResNet'], dash='dash'),
        legendgroup='ResNet',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # Neural ODE trace
    fig.add_trace(go.Scatter(
        x=t_future_np,
        y=pred_future_node[:, i].cpu().numpy(),
        name=f'{labels[i]} Neural ODE',
        mode='lines',
        line=dict(color=colors['Neural ODE'], dash='dash'),
        legendgroup='Neural ODE',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # UDE trace
    fig.add_trace(go.Scatter(
        x=t_future_np,
        y=pred_future_ude[:, i].cpu().numpy(),
        name=f'{labels[i]} UDE',
        mode='lines',
        line=dict(color=colors['UDE'], dash='dash'),
        legendgroup='UDE',
        showlegend=(i == 0)
    ), row=i + 1, col=1)

    # Adding a vertical line at the end of training data
    fig.add_vline(
        x=t.cpu().max(),
        line_width=2,
        line_dash="dash",
        line_color="grey",
        row=i + 1, col=1
    )

fig.update_layout(
    title_text="Previsioni Future dei Modelli (Estrapolazione)",
    height=800,
    width=1400,
    legend_title_text='Legend',
    annotations=[
        dict(
            x=t.cpu().max(), y=1.05,
            xref="x", yref="y",
            text="End of training data",
            showarrow=True,
            arrowhead=2,
            ax=0, ay=-40
        )
    ],
    template='plotly_white',
)

fig.update_xaxes(title_text="Time", row=3, col=1)
fig.update_yaxes(title_text="Fraction of Population", row=1, col=1)
fig.update_yaxes(title_text="Fraction of Population", row=2, col=1)
fig.update_yaxes(title_text="Fraction of Population", row=3, col=1)

fig.show()

## 5. Adding Noise to the Data

In [39]:
noise_std = 0.02
noisy_traj = true_traj + torch.randn_like(true_traj) * noise_std

In [40]:
class NodeNoise(nn.Module):
    def __init__(self, dim=3):
        super().__init__()
        self.res = nn.Sequential(nn.Linear(dim, 64), nn.ReLU(), nn.Linear(64, dim))

    def forward(self, t, y):
        return self.res(y)


class UDENoise(nn.Module):
    def __init__(self, gamma, mu, theta_init=0.01, beta_init=0.9):
        super().__init__()
        # parametri fisici da apprendere
        self.beta = nn.Parameter(torch.tensor(beta_init))
        self.theta = nn.Parameter(torch.tensor(theta_init))
        # gamma e mu fissati esternamente
        self.gamma = gamma
        self.mu = mu
        # rete residua
        self.res = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 3))

    def forward(self, t, y):
        S, I, R = y.unbind(-1)
        # parte analitica
        dS = -self.beta * S * I + self.theta * R
        dI = self.beta * S * I - self.gamma * I - self.mu * I
        dR = self.gamma * I - self.theta * R
        analytic = torch.stack([dS, dI, dR], dim=-1)
        return analytic + self.res(y)

In [41]:
# inizializzo I0 al primo dato, S0 e R0 come parametri da apprendere
noisy_traj = noisy_traj.double().to(device)  # converto in double per maggiore precisione

I_noisy = noisy_traj[:, 1]  # shape: [T]

I0_data = I_noisy[0].clone()
S0 = nn.Parameter(torch.tensor(0.9, dtype=torch.double, device=device))  # valore iniziale guess
R0 = nn.Parameter(torch.tensor(0.05, dtype=torch.double, device=device))  # guess qualsiasi
I0 = I0_data  # fissato al primo dato

y0 = torch.stack([S0, I0_data, R0])

t = t.double()

In [44]:
# istanzio il modello UDE
gamma_val = 1 / 7
mu_val = 1 / 80 / 365
ude_noise = UDENoise(gamma=gamma_val, mu=mu_val, beta_init=0.1, theta_init=0.1).to(device).double()

# ottimizzatore su tutti i parametri: beta, xi, S0, R0 e pesi residui
optimizer = optim.Adam(
    [{'params': ude_noise.parameters()},
     {'params': [S0, R0]}],
    lr=1e-3
)
loss_fn = nn.MSELoss()

In [48]:
# training loop
epochs = 2000
for epoch in tqdm(range(1, epochs + 1), desc='Training UDE with noise', unit='ep'):
    optimizer.zero_grad()

    # costruisco y0 completo
    y0 = torch.stack([S0, I0, R0]).to(device)

    traj = odeint(
        ude_noise, y0, t,
        method='dopri5',
    )

    # seleziono solo la colonna degli infetti
    I_pred = traj[:, 1]

    # loss sul solo I
    loss = loss_fn(I_pred, I_noisy)
    loss.backward()

    torch.nn.utils.clip_grad_norm_(
        list(ude_noise.parameters()) + [S0, R0],
        max_norm=1.0
    )

    optimizer.step()
    with torch.no_grad():
        S0.clamp_(min=1e-6)
        R0.clamp_(min=1e-6)

    if epoch % 5 == 0:
        tqdm.write(f"Epoch {epoch:04d}/{epochs:04d} \t|\t Loss {loss.item():.6f} \t|\t "
                   f"β={ude_noise.beta.item():.4f} \t θ={ude_noise.theta.item():.4f} \t "
                   f"S₀={S0.item():.4f} \t R₀={R0.item():.4f}")

print(f'\nTraining completed!')

Training UDE with noise:   0%|          | 0/2000 [00:00<?, ?ep/s]

Epoch 0005/2000 	|	 Loss 0.000462 	|	 β=0.1251 	 θ=0.1008 	 S₀=0.8527 	 R₀=0.0927
Epoch 0010/2000 	|	 Loss 0.000455 	|	 β=0.1252 	 θ=0.1008 	 S₀=0.8522 	 R₀=0.0929
Epoch 0015/2000 	|	 Loss 0.000451 	|	 β=0.1253 	 θ=0.1008 	 S₀=0.8516 	 R₀=0.0932
Epoch 0020/2000 	|	 Loss 0.000446 	|	 β=0.1254 	 θ=0.1008 	 S₀=0.8508 	 R₀=0.0934
Epoch 0025/2000 	|	 Loss 0.000441 	|	 β=0.1255 	 θ=0.1008 	 S₀=0.8501 	 R₀=0.0937
Epoch 0030/2000 	|	 Loss 0.000437 	|	 β=0.1256 	 θ=0.1007 	 S₀=0.8496 	 R₀=0.0939
Epoch 0035/2000 	|	 Loss 0.000432 	|	 β=0.1257 	 θ=0.1007 	 S₀=0.8489 	 R₀=0.0942
Epoch 0040/2000 	|	 Loss 0.000437 	|	 β=0.1257 	 θ=0.1008 	 S₀=0.8483 	 R₀=0.0945
Epoch 0045/2000 	|	 Loss 0.000439 	|	 β=0.1257 	 θ=0.1007 	 S₀=0.8477 	 R₀=0.0946
Epoch 0050/2000 	|	 Loss 0.000433 	|	 β=0.1258 	 θ=0.1007 	 S₀=0.8465 	 R₀=0.0952
Epoch 0055/2000 	|	 Loss 0.000424 	|	 β=0.1259 	 θ=0.1007 	 S₀=0.8453 	 R₀=0.0958
Epoch 0060/2000 	|	 Loss 0.000419 	|	 β=0.1259 	 θ=0.1007 	 S₀=0.8448 	 R₀=0.0959
Epoch 0065/2000 

KeyboardInterrupt: 

In [49]:
# salvo il modello
torch.save(ude_noise.state_dict(), f'deep_models/noise/UDEnoise EP {epochs} LR {1e-5} LOSS {loss:.4f}.pth')
print(f'Model saved as: deep_models/noise/UDEnoise EP {epochs} LR {1e-5} LOSS {loss:.4f}.pth')

Model saved as: deep_models/noise/UDEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth


In [50]:
# carico il modello
ude_noise.load_state_dict(torch.load(f'deep_models/noise/UDEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth'))

<All keys matched successfully>

In [52]:
# visualizzo i risultati
ude_noise_traj = odeint(
    ude_noise, y0, t,
    method='dopri5').detach()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=true_traj[:, 1].cpu().numpy(),
    mode='lines',
    name='Ground Truth',
    line=dict(color='black', dash='dash')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=ude_noise_traj[:, 1].cpu().numpy(),
    mode='lines',
    name='UDE with Noise',
    line=dict(color='royalblue')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=noisy_traj[:, 1].cpu().numpy(),
    mode='markers',
    name='Noisy Data',
    marker=dict(symbol='cross', color='grey', size=5)
))
fig.update_layout(
    title='UDE with Noise: Infected Trajectory',
    xaxis_title='Time (t)',
    yaxis_title='Infected (I)',
    legend_title='Legend',
    template='plotly_white',
    width=1200,
    height=600
)
fig.show()

In [53]:
# future predictions
future_t = torch.linspace(0., 150., 151).detach()
ude_noise_future = odeint(
    ude_noise, y0, future_t,
    method='dopri5').detach()

true_future_traj_2 = odeint(lambda tt, yy: sirs_deriv(tt, yy), y0, future_t).detach()
true_future_traj_noised = true_future_traj_2 + torch.randn_like(true_future_traj_2) * 0.03

plt.plot(future_t.cpu(), ude_noise_future)
plt.plot(future_t.cpu(), true_future_traj_2, '--')
plt.scatter(future_t.cpu(), true_future_traj_noised[:, 1], marker='o', s=2)
plt.scatter(future_t.cpu(), true_future_traj_noised[:, 0], marker='o', s=2, c='blue')
plt.scatter(future_t.cpu(), true_future_traj_noised[:, 2], marker='o', s=2, c='red')
plt.show()

AssertionError: underflow in dt 5.401981207023612e-17

In [54]:
### Ora invece con la Neural ODE con rumore
node_noisy = NodeNoise(dim=3).to(device).double()
optimizer_noisy = optim.Adam(node_noisy.parameters(), lr=1e-3)
loss_fn_noisy = nn.MSELoss()

In [55]:
# training loop per Neural ODE con rumore
epochs_noisy = 2000
for epoch in tqdm(range(1, epochs_noisy + 1), desc='Training Neural ODE with noise', unit='ep'):
    optimizer_noisy.zero_grad()

    # costruisco y0 completo
    y0 = torch.stack([S0, I0_data, R0]).to(device)

    traj_noisy = odeint(
        node_noisy, y0, t,
        method='dopri5',
    )

    # seleziono solo la colonna degli infetti
    I_pred_noisy = traj_noisy[:, 1]

    # loss sul solo I
    loss_noisy = loss_fn_noisy(I_pred_noisy, I_noisy)
    loss_noisy.backward()

    torch.nn.utils.clip_grad_norm_(
        node_noisy.parameters(),
        max_norm=1.0
    )

    optimizer_noisy.step()

    if epoch % 5 == 0:
        tqdm.write(f"Epoch {epoch:04d}/{epochs_noisy:04d} \t|\t Loss {loss_noisy.item():.6f}")
print(f'\nTraining completed!')

Training Neural ODE with noise:   0%|          | 0/2000 [00:00<?, ?ep/s]

Epoch 0005/2000 	|	 Loss 1952.437375
Epoch 0010/2000 	|	 Loss 3.321033
Epoch 0015/2000 	|	 Loss 0.196044
Epoch 0020/2000 	|	 Loss 0.038201
Epoch 0025/2000 	|	 Loss 0.014605
Epoch 0030/2000 	|	 Loss 0.009906
Epoch 0035/2000 	|	 Loss 0.002830
Epoch 0040/2000 	|	 Loss 0.003322
Epoch 0045/2000 	|	 Loss 0.001525
Epoch 0050/2000 	|	 Loss 0.001250
Epoch 0055/2000 	|	 Loss 0.000957
Epoch 0060/2000 	|	 Loss 0.000966
Epoch 0065/2000 	|	 Loss 0.000757
Epoch 0070/2000 	|	 Loss 0.000700
Epoch 0075/2000 	|	 Loss 0.000661
Epoch 0080/2000 	|	 Loss 0.000610
Epoch 0085/2000 	|	 Loss 0.000589
Epoch 0090/2000 	|	 Loss 0.000569
Epoch 0095/2000 	|	 Loss 0.000559
Epoch 0100/2000 	|	 Loss 0.000550
Epoch 0105/2000 	|	 Loss 0.000538
Epoch 0110/2000 	|	 Loss 0.000528
Epoch 0115/2000 	|	 Loss 0.000520
Epoch 0120/2000 	|	 Loss 0.000512
Epoch 0125/2000 	|	 Loss 0.000505
Epoch 0130/2000 	|	 Loss 0.000498
Epoch 0135/2000 	|	 Loss 0.000491
Epoch 0140/2000 	|	 Loss 0.000485
Epoch 0145/2000 	|	 Loss 0.000479
Epoch 0150/

In [59]:
torch.save(node_noisy.state_dict(), f'deep_models/noise/NODEnoise EP {epochs} LR {1e-5} LOSS {loss_noisy:.4f}.pth')
print(f'Model saved as: deep_models/noise/NODEnoise EP {epochs} LR {1e-5} LOSS {loss_noisy:.4f}.pth')

Model saved as: deep_models/noise/NODEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth


In [61]:
# carico il modello
node_noisy.load_state_dict(torch.load(f'deep_models/noise/NODEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth'))

node_noisy_traj = odeint(
    node_noisy, y0, t,
    method='dopri5').detach()

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=true_traj[:, 1].cpu().numpy(),
    mode='lines',
    name='Ground Truth',
    line=dict(color='black', dash='dash')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=node_noisy_traj[:, 1].cpu().numpy(),
    mode='lines',
    name='Neural ODE with Noise',
    line=dict(color='firebrick')
))
fig.add_trace(go.Scatter(
    x=t.cpu().numpy(),
    y=noisy_traj[:, 1].cpu().numpy(),
    mode='markers',
    name='Noisy Data',
    marker=dict(symbol='cross', color='grey', size=5)
))
fig.update_layout(
    title='Neural ODE with Noise: Infected Trajectory',
    xaxis_title='Time (t)',
    yaxis_title='Infected (I)',
    legend_title='Legend',
    template='plotly_white',
    width=1200,
    height=600
)
fig.show()

In [63]:
# 1. Carico i modelli salvati
# Assicurati che i file .pth siano nella stessa cartella o fornisci il percorso completo.
ude_noise.load_state_dict(torch.load(f'deep_models/noise/UDEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth'))
node_noisy.load_state_dict(torch.load(f'deep_models/noise/NODEnoise EP 2000 LR 1e-05 LOSS 0.0004.pth'))

# Imposta i modelli in modalità di valutazione
ude_noise.eval()
node_noisy.eval()

# 2. Genero le traiettorie per entrambi i modelli
with torch.no_grad():  # Disabilita il calcolo del gradiente per l'inferenza
    ude_noise_traj = odeint(ude_noise, y0, t, method='dopri5').detach()
    node_noisy_traj = odeint(node_noisy, y0, t, method='dopri5').detach()

# 3. Converto i tensori in array NumPy per Plotly e sposto su CPU se necessario
t_np = t.cpu().numpy()
true_traj_np = true_traj.cpu().numpy()
noisy_traj_np = noisy_traj.cpu().numpy()
ude_noise_traj_np = ude_noise_traj.cpu().numpy()
node_noisy_traj_np = node_noisy_traj.cpu().numpy()

In [64]:
# 4. Creo la figura con Plotly
fig = go.Figure()

# Aggiungo la traiettoria reale (linea tratteggiata)
fig.add_trace(go.Scatter(
    x=t_np,
    y=true_traj_np[:, 1],
    mode='lines',
    name='Traiettoria Reale',
    line=dict(color='black', dash='dash')
))

# Aggiungo la traiettoria predetta dal modello UDE (linea continua)
fig.add_trace(go.Scatter(
    x=t_np,
    y=ude_noise_traj_np[:, 1],
    mode='lines',
    name='Predizione UDE',
    line=dict(color='royalblue')
))

# Aggiungo la traiettoria predetta dal modello NODE (linea continua)
fig.add_trace(go.Scatter(
    x=t_np,
    y=node_noisy_traj_np[:, 1],
    mode='lines',
    name='Predizione NODE',
    line=dict(color='firebrick')
))

# Aggiungo i dati rumorosi (punti a croce)
fig.add_trace(go.Scatter(
    x=t_np,
    y=noisy_traj_np[:, 1],
    mode='markers',
    name='Dati Rumorosi',
    marker=dict(symbol='cross', color='grey', size=5)
))

# 5. Personalizzo il layout del grafico
fig.update_layout(
    title='Ricostruzione classe degli infetti con dati rumorosi',
    xaxis_title='Tempo (t)',
    yaxis_title='Infected (I)',
    legend_title='Legenda',
    template='plotly_white'  # Un template pulito per la visualizzazione
)

# 6. Mostro il grafico
fig.show()

In [65]:
# future predictions
future_t = torch.linspace(0., 200., 201).detach()
node_noisy_future = odeint(
    node_noisy, y0, future_t,
    method='dopri5').detach()
true_future_traj_2 = odeint(lambda tt, yy: sirs_deriv(tt, yy), y0, future_t).detach()
true_future_traj_noised = true_future_traj_2 + torch.randn_like(true_future_traj_2) * 0.03
plt.plot(future_t.cpu(), node_noisy_future[:, 1])
plt.plot(future_t.cpu(), true_future_traj_2[:, 1], '--')
plt.scatter(future_t.cpu(), true_future_traj_noised[:, 1], marker='o', s=2)
# plt.scatter(future_t.cpu(), true_future_traj_noised[:, 0], marker='o', s=2, c='blue')
# plt.scatter(future_t.cpu(), true_future_traj_noised[:, 2], marker='o', s=2, c='red')
plt.show()

AssertionError: underflow in dt 5.401981207023612e-17